# Causal Inference for Credit Limit Experiments

**AuditLend Intelligence Core (ALICe)**  
Phase 9 — Estimating the causal impact of credit-limit increases on default rates.

## Motivation

Simple before/after comparisons confound treatment effects with time trends.  
Borrowers who receive limit increases differ systematically from those who don't (selection bias).

We apply three causal methods to disentangle correlation from causation:

| Method | Question | Key Assumption |
|--------|----------|----------------|
| **Propensity Score Matching** | What is the ATT of a limit increase? | Unconfoundedness given observables |
| **Synthetic Control** | What would have happened to a treated portfolio without the treatment? | Pre-treatment trajectory is informative |
| **Difference-in-Differences** | How do default rates change relative to a control group? | Parallel trends |

> **Caveat**: These are observational methods. Causal claims require domain
> justification for the identifying assumptions.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '..')
from ml.causal.psm import PropensityScoreMatcher, compute_balance
from ml.causal.synthetic_control import SyntheticControl

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded.')

## Section 1: Propensity Score Matching (PSM)

We simulate a credit-portfolio experiment where some borrowers receive a limit increase (treatment) and others do not (control). Borrowers are not randomly assigned — treatment correlates with income and credit score.

PSM estimates the **Average Treatment Effect on the Treated (ATT)**:  
*What is the causal impact of a limit increase on default rates for those who received it?*

In [ ]:
np.random.seed(42)
N = 500

income = np.random.lognormal(mean=11.5, sigma=0.6, size=N)
credit_score = np.clip(np.random.normal(700, 80, size=N), 350, 850)
dti = np.clip(np.random.normal(0.35, 0.12, size=N), 0.0, 1.0)
age = np.random.randint(22, 70, size=N)

logit_treatment = (
    -3.0
    + 0.15 * (income / 10000)
    + 0.008 * (credit_score - 600)
    - 2.0 * dti
    + 0.01 * (age - 35)
)
prob_treatment = 1 / (1 + np.exp(-logit_treatment))
treatment = np.random.binomial(1, prob_treatment)

default_prob = 0.3 - 0.15 * treatment - 0.0004 * (credit_score - 600) + 0.3 * dti + 0.1 * (income < 30000)
default_prob = np.clip(default_prob, 0.01, 0.99)
default = np.random.binomial(1, default_prob)

df = pd.DataFrame({
    'income': income,
    'credit_score': credit_score,
    'dti': dti,
    'age': age,
    'treatment': treatment,
    'default': default,
})
df['log_income'] = np.log(df['income'])

print(f'Treated: {treatment.sum()}, Control: {N - treatment.sum()}')
print(f'Raw default rate - Treated: {df[df.treatment==1].default.mean():.3f}')
print(f'Raw default rate - Control: {df[df.treatment==0].default.mean():.3f}')
print(f'Naive difference: {df[df.treatment==1].default.mean() - df[df.treatment==0].default.mean():.3f}')

In [ ]:
features = ['log_income', 'credit_score', 'dti', 'age']

treated = df[df.treatment == 1].copy()
control = df[df.treatment == 0].copy()

matcher = PropensityScoreMatcher(caliper=0.05)
result = matcher.match(
    treatment_features=treated[features].to_dict('records'),
    control_features=control[features].to_dict('records'),
    treatment_outcomes=treated['default'].tolist(),
    control_outcomes=control['default'].tolist(),
    treatment_ids=treated.index.astype(str).tolist(),
    control_ids=control.index.astype(str).tolist(),
)

print(f'Matched pairs: {result.n_matched}')
print(f'ATT (limit increase on default rate): {result.att:.4f}')
print(f'95% CI: ({result.att_ci[0]:.4f}, {result.att_ci[1]:.4f})')
print()
print('Naive difference (biased): '
      f'{treated["default"].mean() - control["default"].mean():.4f}')

### Covariate Balance Check

Standardized mean differences should be close to zero after matching (|std_diff| < 0.1 is the rule of thumb).

In [ ]:
balance = result.balance_statistics
balance_df = pd.DataFrame(balance).T
display(balance_df.style.format('{:.4f}'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

balance_before = compute_balance(
    treated[features].to_dict('records'),
    control[features].to_dict('records'),
)

names = list(balance.keys())
before_vals = [balance_before[n]['std_diff'] for n in names]
after_vals = [balance[n]['std_diff'] for n in names]

axes[0].barh(names, before_vals, color='crimson', alpha=0.7)
axes[0].axvline(0, color='k', linewidth=0.5)
axes[0].axvline(-0.1, color='gray', linestyle='--', linewidth=0.8)
axes[0].axvline(0.1, color='gray', linestyle='--', linewidth=0.8)
axes[0].set_title('Before Matching')
axes[0].set_xlabel('Standardized Mean Difference')

axes[1].barh(names, after_vals, color='forestgreen', alpha=0.7)
axes[1].axvline(0, color='k', linewidth=0.5)
axes[1].axvline(-0.1, color='gray', linestyle='--', linewidth=0.8)
axes[1].axvline(0.1, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_title('After Matching')
axes[1].set_xlabel('Standardized Mean Difference')

plt.tight_layout()
plt.show()

print('Balance improved after matching: std_diff values shrink toward zero.')

## Section 2: Synthetic Control

Build a synthetic version of a treated portfolio segment using a weighted combination of untreated segments. Compare post-treatment outcomes to estimate the causal effect.

In [ ]:
np.random.seed(123)
T_pre = 12
T_post = 6

time = np.arange(T_pre + T_post)

control_units = {
    'Segment_A': 100 + 2 * time + np.random.normal(0, 3, T_pre + T_post),
    'Segment_B': 95 + 1.8 * time + np.random.normal(0, 4, T_pre + T_post),
    'Segment_C': 105 + 2.2 * time + np.random.normal(0, 3.5, T_pre + T_post),
    'Segment_D': 90 + 2.1 * time + np.random.normal(0, 5, T_pre + T_post),
}

treated_observed = 100 + 2 * time + np.random.normal(0, 3, T_pre + T_post)
treated_observed[T_pre:] += -8

control_pre = {k: v[:T_pre].tolist() for k, v in control_units.items()}
control_post = {k: v[T_pre:].tolist() for k, v in control_units.items()}
treated_pre = treated_observed[:T_pre].tolist()
treated_post = treated_observed[T_pre:].tolist()

sc = SyntheticControl(unit_id='Treated_Portfolio')
result_sc = sc.fit(treated_pre, control_pre, treated_post, control_post)

print('Synthetic Control Weights:')
for unit, weight in sorted(result_sc.weights.items()):
    print(f'  {unit}: {weight:.4f}')
print(f'\nPre-treatment RMSE: {result_sc.pre_treatment_rmse:.4f}')
print(f'Post-treatment causal effect: {result_sc.causal_effect:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(time, result_sc.observed_outcomes, 'o-', color='black',
        label='Treated (Observed)', linewidth=2)
ax.plot(time, result_sc.synthetic_outcomes, 's--', color='steelblue',
        label='Synthetic Control', linewidth=2)
ax.axvline(T_pre - 0.5, color='red', linestyle=':', linewidth=1.2,
           label='Treatment Start')

ax.fill_between(time[T_pre:],
                 result_sc.observed_outcomes[T_pre:],
                 result_sc.synthetic_outcomes[T_pre:],
                 color='steelblue', alpha=0.15,
                 label=f'Causal Effect = {result_sc.causal_effect:.2f}')

ax.set_xlabel('Time Period')
ax.set_ylabel('Default Rate (bps)')
ax.set_title('Synthetic Control: Treated Portfolio vs Counterfactual')
ax.legend()
plt.tight_layout()
plt.show()

print(f'The treated portfolio defaults {result_sc.causal_effect:.1f} bps '
      f"{'higher' if result_sc.causal_effect > 0 else 'lower'} than the synthetic counterfactual.")

## Section 3: Difference-in-Differences (DiD)

Simple panel DiD: two groups (limit increase vs no increase), two time periods (pre/post).  
The treatment effect is the difference in the group-specific changes over time.

$$ \text{DiD} = (\bar{Y}_{treat,post} - \bar{Y}_{treat,pre}) - (\bar{Y}_{control,post} - \bar{Y}_{control,pre}) $$

In [ ]:
np.random.seed(456)

n_per_group = 200
base_default_rate = 0.12
time_trend = 0.015
treatment_effect = -0.04

groups = ['control'] * n_per_group + ['treatment'] * n_per_group
periods = ['pre', 'post']

data = []
for group in ['control', 'treatment']:
    for period in ['pre', 'post']:
        mu = base_default_rate
        if period == 'post':
            mu += time_trend
        if group == 'treatment' and period == 'post':
            mu += treatment_effect
        defaults = np.random.binomial(1, mu, n_per_group)
        for d in defaults:
            data.append({'group': group, 'period': period, 'default': d})

did_df = pd.DataFrame(data)

summary = did_df.groupby(['group', 'period'])['default'].agg(['mean', 'std', 'count']).reset_index()
pivot = summary.pivot_table(index='group', columns='period', values='mean', aggfunc='first')
display(pivot.style.format('{:.4f}'))

did_estimate = (pivot.loc['treatment', 'post'] - pivot.loc['treatment', 'pre']) - (
    pivot.loc['control', 'post'] - pivot.loc['control', 'pre']
)
print(f'DiD estimate (causal effect on default rate): {did_estimate:.4f}')
print(f'True treatment effect: {treatment_effect:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

means = did_df.groupby(['group', 'period'])['default'].mean().unstack()

for group, color, marker in [('control', 'gray', 'o'), ('treatment', 'crimson', 's')]:
    row = means.loc[group]
    ax.plot(['Pre', 'Post'], row.values, f'{marker}-', color=color,
            label=group, linewidth=2, markersize=8)

control_change = means.loc['control', 'post'] - means.loc['control', 'pre']
treatment_change = means.loc['treatment', 'post'] - means.loc['treatment', 'pre']

ax.annotate(f'Control Δ = {control_change:+.3f}',
            xy=(1, means.loc['control', 'post']), xytext=(1.25, means.loc['control', 'post'] + 0.005),
            arrowprops=dict(arrowstyle='->'), fontsize=9)
ax.annotate(f'Treatment Δ = {treatment_change:+.3f}',
            xy=(1, means.loc['treatment', 'post']), xytext=(1.25, means.loc['treatment', 'post'] + 0.005),
            arrowprops=dict(arrowstyle='->'), fontsize=9)

ax.set_ylabel('Default Rate')
ax.set_title(f'Difference-in-Differences\nEstimated Effect = {did_estimate:.4f}')
ax.legend()
plt.tight_layout()
plt.show()

## Key Findings

| Method | Estimate | Interpretation |
|--------|----------|----------------|
| **PSM (ATT)** | See Section 1 | Limit increase reduces/default increases default rate by X pp among the treated |
| **Synthetic Control** | See Section 2 | Treated portfolio defaults are Y bps lower/higher than the counterfactual |
| **Difference-in-Differences** | See Section 3 | Default rates change by Z pp relative to the control group |

### Interpretation Notes

1. **PSM** adjusts for observable selection bias. The ATT is closer to the true causal effect than the naive comparison, but depends on unconfoundedness.
2. **Synthetic Control** works best when a few comparison units closely track the treated unit's pre-treatment trajectory. The gap after treatment is plausibly causal.
3. **DiD** removes time-invariant differences between groups but requires parallel trends in the absence of treatment.

### Limitations

- These estimates are from synthetic data and should not be used for real lending decisions without validation.
- Real-world limit increases are rarely as clean as simulated experiments; regression discontinuity and instrumental variables may complement these methods.
- The parallel-trends and unconfoundedness assumptions are fundamentally untestable; sensitivity analysis (e.g., Rosenbaum bounds) is recommended before deployment.